# Post-OCR Correction for Sanskrit (Devanagari)
**Option 3: AI/ML Model Training assignment**

Pipeline: real OCR baseline (Tesseract `san`) → error analysis → synthetic OCR-noise generation → ByT5-small fine-tune → evaluation on real scans.

Runtime: Google Colab, **T4 GPU** (Runtime → Change runtime type → T4).

**Links**
- Trained model (Hugging Face, public): https://huggingface.co/MrunalYewatkar45/byt5-small-sanskrit-post-ocr
- Code, data and all results: https://github.com/Mrunal31-Stark/OCR_Extraction_indic_text (`REPORT.md`, `README.md` → Setup)

> **Training status:** training was **stopped early, at step 1,768 of 2,125 (83% of one epoch)**, because the **Google Colab GPU runtime kept disconnecting**. The released model is the best saved checkpoint, **`checkpoint-1600`** (dev CER 5.2%, still improving). See cell 3d and the report.

**How to use this notebook:** to only *use or evaluate* the trained model, jump to **Phase 5**, or follow README Option A; no training is needed. To reproduce everything, run Phases 0 → 4 in order (~2.5 h on a T4). Phase 2 can be skipped with the ready-made splits in the repo (see the note in Phase 2).

## Phase 0: Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get -qq install -y tesseract-ocr tesseract-ocr-san tesseract-ocr-hin > /dev/null
!pip -q install jiwer pytesseract
!tesseract --version | head -1 && tesseract --list-langs

name, memory.total [MiB]
Tesla T4, 15360 MiB
tesseract 5.3.4
List of available languages in "/usr/share/tesseract-ocr/5/tessdata/" (4):
eng
hin
osd
san

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive

## Phase 1: Real OCR baseline + error analysis
The 5 pages in `Data/` (scan-style images + ground-truth `.md`) are a **held-out real test set**. They are never used for training.

First we measure how badly off-the-shelf Tesseract does. Then we record *which* errors it makes, so the synthetic noise in Phase 2 copies real OCR behaviour instead of guesses.

The data is pulled from the project repo, so the notebook runs end-to-end with no manual uploads.

In [ ]:
import os, re, json, unicodedata, difflib
from collections import Counter
from PIL import Image, ImageOps
import pytesseract, jiwer

REPO = '/content/OCR_Extraction_indic_text'
if not os.path.exists(REPO):
    !git clone -q https://github.com/Mrunal31-Stark/OCR_Extraction_indic_text.git {REPO}
DATA, WORK = f'{REPO}/Data', '/content/work'
os.makedirs(WORK, exist_ok=True)

def norm(s):
    """Unicode NFC, drop ZWJ/ZWNJ, collapse spaces, drop empty lines. Applied to BOTH sides before any metric."""
    s = unicodedata.normalize('NFC', s).replace('‌', '').replace('‍', '')
    s = re.sub(r'[ \t]+', ' ', s)
    return '\n'.join(l.strip() for l in s.strip().splitlines() if l.strip())

def flat(s):  # page-level metrics ignore line breaks (OCR line segmentation can differ)
    return ' '.join(s.split())

gt = {i: norm(open(f'{DATA}/page_{i}.md', encoding='utf-8').read()) for i in range(1, 6)}
print(gt[1])

अथ आयुर्वेदः नाम शास्त्रम्
आयुर्वेदः नाम आयुः वेदयति इति आयुर्वेदः
आयुः नाम शरीर इन्द्रिय सत्त्व आत्म
संयोगः तस्य हिताहितं सुखं दुःखम् आयुः
तस्य हितं च अहितं च मानम् च तच् च
यत्र उक्तं तत् आयुर्वेदः

In [ ]:
# OCR grid: {raw, binarized} x {san, hin}. Pick the best config as "the OCR system" whose output we must correct.
def binarize(img):
    g = ImageOps.autocontrast(img.convert('L'))
    return g.point(lambda p: 255 if p > 140 else 0)  # ponytail: fixed threshold, Otsu if pages vary in lighting

imgs = {i: Image.open(f'{DATA}/page_{i}.png') for i in range(1, 6)}
ocr, rows = {}, []
for prep in ('raw', 'bin'):
    for lang in ('san', 'hin'):
        key = f'{prep}-{lang}'
        ocr[key] = {i: norm(pytesseract.image_to_string(imgs[i] if prep == 'raw' else binarize(imgs[i]), lang=lang, config='--psm 6')) for i in imgs}
        refs, hyps = [flat(gt[i]) for i in imgs], [flat(ocr[key][i]) for i in imgs]
        rows.append((key, jiwer.cer(refs, hyps), jiwer.wer(refs, hyps)))

print(f"{'config':10} {'CER':>7} {'WER':>7}")
for k, c, w in sorted(rows, key=lambda r: r[1]):
    print(f'{k:10} {c:7.3f} {w:7.3f}')
BEST = min(rows, key=lambda r: r[1])[0]
print('\nbest config:', BEST)
for i in imgs:
    print(f'\n--- page {i} ---\nGT : {flat(gt[i])}\nOCR: {flat(ocr[BEST][i])}')

config         CER     WER
raw-san      0.096   0.339
bin-san      0.099   0.348
raw-hin      0.106   0.409
bin-hin      0.119   0.461

best config: raw-san

--- page 1 ---
GT : अथ आयुर्वेदः नाम शास्त्रम् आयुर्वेदः नाम आयुः वेदयति इति आयुर्वेदः आयुः नाम शरीर इन्द्रिय सत्त्व आत्म संयोगः तस्य हिताहितं सुखं दुःखम् आयुः तस्य हितं च अहितं च मानम् च तच् च यत्र उक्तं तत् आयुर्वेदः
OCR: अथ आयुर्वेदः नाम शास्त्रम् आयुर्वेदः नाम आगुः ठेट्रमति इति आयुर्वेद आरूः नाम शरीर इन्दिम सत्त्व आत्म संयोगः तस्त्र हिताहितं युखं दृखम् आयुः तस्त्र हितं च अदितं च मानम् च तच् च मत्र उक्तं तत् आयुर्वेद

--- page 2 ---
GT : त्रयो दोषाः वातः पित्तं कफः इति एते दोषाः शरीरस्य धारणम् कुर्वन्ति वातः गतिः कारणम् पित्तं पाक कारणम् कफः स्थिरत्व कारणम्
OCR: त्रमो द्रोषाः वातः पित्तं कफः इति एते द्रोणाः शरीरस्य धारणम् कुर्वन्ति तातः गतिः कारणम् पित्तं पाक कारणम् कफः स्थिरत्व कारणम्

--- page 3 ---
GT : आहारः महत्त्वं आयुः वर्धनम् करोति सम्यक् आहारः शरीरं बलम् वर्धयति असम्यक् आहारः रोग कारणम् भवति हिताहार सेवनं आयुः वर्धयति


In [ ]:
# Character-level error analysis: align GT vs OCR per page and count edit operations.
# These counts calibrate the synthetic noise generator in Phase 2.
def name(s):
    return '+'.join(unicodedata.name(c, hex(ord(c))).replace('DEVANAGARI ', '') for c in s) or '∅'

ops = Counter()
for i in imgs:
    a, b = flat(gt[i]), flat(ocr[BEST][i])
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b, autojunk=False).get_opcodes():
        if tag != 'equal':
            ops[(tag, a[i1:i2], b[j1:j2])] += 1

by_type = Counter()
for (tag, src, dst), n in ops.items():
    by_type[tag] += n
print('edit ops by type:', dict(by_type))
print(f"\n{'n':>3}  {'op':8} {'GT':>8} -> {'OCR':8}  names")
for (tag, src, dst), n in ops.most_common(30):
    print(f'{n:3}  {tag:8} {src!r:>8} -> {dst!r:8}  {name(src)} -> {name(dst)}')

json.dump({'best_config': BEST, 'gt': gt, 'ocr': ocr,
           'ops': [[t, s, d, n] for (t, s, d), n in ops.most_common()]},
          open(f'{WORK}/phase1.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

edit ops by type: {'replace': 38, 'delete': 8, 'insert': 5}

  n  op             GT -> OCR       names
  8  replace       'य' -> 'म'       LETTER YA -> LETTER MA
  5  replace       'ं' -> 'े'       SIGN ANUSVARA -> VOWEL SIGN E
  3  replace       'व' -> 'ठ'       LETTER VA -> LETTER TTHA
  3  delete        'ः' -> ''        SIGN VISARGA -> ∅
  3  replace       'स' -> 'य'       LETTER SA -> LETTER YA
  3  replace       'ह' -> 'द'       LETTER HA -> LETTER DA
  3  insert         '' -> '्र'      ∅ -> SIGN VIRAMA+LETTER RA
  2  replace       'य' -> 'ग'       LETTER YA -> LETTER GA
  2  replace       'य' -> 'त्र'     LETTER YA -> LETTER TA+SIGN VIRAMA+LETTER RA
  2  replace       'ष' -> 'ण'       LETTER SSA -> LETTER NNA
  2  replace       'र' -> 'य'       LETTER RA -> LETTER YA
  1  replace      'दय' -> 'ट्रम'    LETTER DA+LETTER YA -> LETTER TTA+SIGN VIRAMA+LETTER RA+LETTER MA
  1  replace      'यु' -> 'रू'      LETTER YA+VOWEL SIGN U -> LETTER RA+VOWEL SIGN UU
  1  delete       '्र' -> ''

In [ ]:
# Sanity checks for Phase 0/1: fail loudly if anything is off.
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

# norm(): NFC must merge decomposed forms, strip ZWJ/ZWNJ, collapse spaces
assert norm('क़') == norm('क़'), 'NFC/nukta mismatch'
assert norm('क्‍ष  त') == 'क्ष त'
assert all(norm(gt[i]) == gt[i] for i in gt), 'GT not idempotent under norm'

# phase1.json round-trips and matches in-memory results
p1 = json.load(open(f'{WORK}/phase1.json', encoding='utf-8'))
assert p1['best_config'] == BEST and len(p1['gt']) == 5 and sum(o[3] for o in p1['ops']) == sum(ops.values())

# OCR is deterministic (same page twice -> same text), so the baseline is reproducible
assert norm(pytesseract.image_to_string(imgs[1], lang='san', config='--psm 6')) == ocr[BEST][1]

# metric sanity: identical -> 0, our flat() doesn't hide line-content errors
assert jiwer.cer(flat(gt[1]), flat(gt[1])) == 0
print(f"OK | baseline {BEST}: CER {dict((r[0], r[1]) for r in rows)[BEST]:.3f} | {sum(ops.values())} edit ops | GT chars {sum(len(flat(g)) for g in gt.values())}")

GPU: Tesla T4 | torch 2.14.1+cu130
OK | baseline raw-san: CER 0.096 | 51 edit ops | GT chars 705

### Phase 1 findings
**Baseline (best = raw image + Tesseract `san`, `--psm 6`): CER 9.6%, WER 33.9%** on 5 pages (705 chars). This is the "before" number the corrector has to beat.

- Binarization didn't help (the page background is already clean). The `hin` model is worse because its Hindi language model pulls toward Hindi spellings.
- A 10% CER gives a 34% WER because one wrong character ruins the whole word. So **word-level correction is where the value is**.

**What real OCR gets wrong** (51 edits: 38 substitutions, 8 deletions, 5 insertions):
1. **Look-alike consonant swaps, which depend on the font (~60% of errors):** य→म (8×), व→ठ/त/क/न, ह→द, स→य, ष→ण, र→य.
2. **Diacritic (matra/mark) errors:** anusvara `ं`→`े` (5×, both sit above the line), visarga `ः` dropped (3×), dropped or reduced vowel signs (`ौ`, `ा`, `ै`→`े`).
3. **Conjunct (half-letter) errors:** spurious `्र` inserted (दोष→द्रोष), conjuncts lost (स्य→स, त्त्व→त्व).
4. **Real-word errors, the hardest case:** रोगः→**योगः**, दोष→**द्रोण**, विहारः→**निहार**. The OCR output is itself a *valid* Sanskrit word, so a dictionary lookup can't catch it and only sentence context can fix it. This is why we use a seq2seq model rather than a spell-checker.

**How this shapes Phase 2:** the synthetic noise is weighted toward (1) a confusion table of look-alike glyphs, then (2) mark drops and swaps, then (3) conjunct insertions and drops, at roughly the same ratios. Word splits and merges weren't seen here, so they get a low probability.

**Limitations of Phase 1**
- **The test set is tiny:** 5 pages, 705 characters, 51 edits. One fixed word moves CER by about 0.5 points, so small differences in Phase 4 are within noise. We report raw counts next to every percentage.
- **The pages aren't truly old scans.** They use one clean, scan-*style* typeface on uniform paper, with no bleed-through, broken glyphs, skew or multi-column layout. Real manuscript OCR will be much worse and make different errors.
- **We only tested one OCR engine** (Tesseract 5, `--psm 6`) and a small preprocessing grid. Other engines (Google Vision, IndicOCR) make different mistakes, so the corrector is tuned to Tesseract's habits.
- **The error counts are approximate.** `difflib` aligns Unicode code points, not graphemes (visible characters), so one visual mistake on a conjunct can count as 1–3 edits, and the category ratios are rough.
- **The error analysis was done on the test pages themselves.** Only the error-type ratios and the confusion pairs flow into Phase 2, never the text. Still, this uses test-set knowledge, and real-page results may be somewhat optimistic.

## Phase 2: Training data
> **Shortcut (skip Phase 2, ~25 min):** the generated splits are in the repo's `datasets/` folder. Instead of running 2a–2d, run:
> ```
> !mkdir -p /content/drive/MyDrive/sanskrit_ocr /content/work
> !cp /content/OCR_Extraction_indic_text/datasets/* /content/drive/MyDrive/sanskrit_ocr/
> !cp /content/OCR_Extraction_indic_text/datasets/* /content/work/
> ```
> and continue with Phase 3.

**2a. Clean Sanskrit corpus.** Sanskrit Wikipedia (`wikimedia/wikipedia`, `20231101.sa`). We cut it into line-sized chunks of 3–10 words, shaped like the test lines, and keep only chunks that are ≥90% Devanagari. We then remove duplicates and **remove any chunk that overlaps the 5 test pages**.

In [ ]:
import random
from datasets import load_dataset
random.seed(0)

DEVA = re.compile(r'[ऀ-ॿ]')
def deva_ratio(s):
    chars = s.replace(' ', '')
    return len(DEVA.findall(chars)) / max(1, len(chars))

wiki = load_dataset('wikimedia/wikipedia', '20231101.sa', split='train')
print('articles:', len(wiki))

chunks = set()
for text in wiki['text']:
    for para in norm(text).split('\n'):
        for sent in re.split(r'(?<=[।॥])\s*', para):          # keep the danda with its sentence
            words = sent.split()
            i = 0
            while i < len(words):
                n = random.randint(3, 10)                       # line-sized chunks like the test pages
                c = ' '.join(words[i:i + n]); i += n
                if 15 <= len(c) <= 120 and deva_ratio(c) >= 0.9 and not re.search(r'[0-9a-zA-Z]', c):
                    chunks.add(c)

# Leakage guard: drop any chunk containing a 3-word window from a test page
test_trigrams = {' '.join(w[k:k + 3]) for g in gt.values() for w in [flat(g).split()] for k in range(len(w) - 2)}
before = len(chunks)
chunks = sorted(c for c in chunks if not any(t in c for t in test_trigrams))
print(f'clean chunks: {len(chunks)} (removed {before - len(chunks)} overlapping test pages)')
for c in random.sample(chunks, 8): print('  ', c)

articles: 12156
clean chunks: 436056 (removed 0 overlapping test pages)
   वर्धमानस्य कस्यचित् वृक्षविशेषस्य फलम्
   अथर्ववेदस्य गोपथब्राह्मणस्य एकस्याः कथानुसारं, देवराजः
   - सनन्दन- सनत्कुमाराणां नामान्युद्धृतानि ।
   अयं मे अतिथिः इत्युक्तम् ।
   परम्परया तदस्त्रं भगदत्तेन लब्धम्।
   एतस्मात् दिनात् वर्षान्ताय ६९ दिनानि अवशिष्टानि ।
   परन्तु मन्षुयस्तु भवत्कृपया विशेषविवेकशक्तेः स्वामी भवति ।
   द्वादशभाव सम्बन्धिताम्बूलानि गणयन्ति ।

**2b. Synthetic OCR noise (`noise.py`).** Rule-based corruption calibrated on the Phase 1 error mix: look-alike consonant swaps 60%, mark errors 25%, conjunct insert/drop 10%, spacing/punctuation 5%. 10% of lines stay clean, so the model learns to *leave correct text alone* (this limits over-correction). Below we check that the generated errors have the same mix as real Tesseract errors.

In [ ]:
%%writefile /content/noise.py
"""Synthetic OCR noise for Devanagari, calibrated on real Tesseract errors (see notebook Phase 1).

Error mix in real OCR on our test pages: ~60% look-alike consonant swaps, ~25% mark (matra/anusvara/visarga)
errors, ~10% conjunct insert/drop, ~5% spacing/punctuation. `corrupt()` reproduces that mix.
"""
import random
import unicodedata

# Look-alike glyph confusions. First entries were observed in Phase 1; the rest are well-known Devanagari OCR confusions.
CONSONANT_CONFUSIONS = {
    'य': ['म', 'थ', 'ग', 'त्र'], 'व': ['ठ', 'त', 'क', 'न', 'ब'], 'ह': ['द'], 'स': ['य', 'म'], 'ष': ['ण', 'प'],
    'र': ['य'], 'म': ['य', 'भ'], 'द': ['ह', 'ढ', 'द्र'], 'प': ['ष', 'य'], 'ब': ['व'], 'घ': ['ध'], 'ध': ['घ'],
    'भ': ['म'], 'ट': ['ठ'], 'ठ': ['ट', 'व'], 'न': ['त'], 'त': ['न'], 'ण': ['ष'], 'ख': ['रव'], 'थ': ['य'],
    'ञ': ['ज'], 'झ': ['भ'], 'ङ': ['ड'], 'ड': ['ङ'], 'छ': ['ढ'], 'क': ['फ'], 'फ': ['क'],
}
MARK_CONFUSIONS = {
    'ं': ['े', ''], 'ः': ['', ':'], 'ा': [''], 'ि': ['ी', ''], 'ी': ['ि'], 'ु': ['ू', ''], 'ू': ['ु'],
    'े': ['ै', ''], 'ै': ['े'], 'ो': ['ौ', 'ा'], 'ौ': ['ो', ''], 'ृ': ['ु'], '्': [''], 'ँ': ['ं'],
}
PUNCT_CONFUSIONS = {'।': ['|', '', 'l'], '॥': ['||', '।', '|'], ' ': ['']}  # ' ' -> '' merges words
CONSONANTS = set('कखगघङचछजझञटठडढणतथदधनपफबभमयरलवशषसह')

# op -> relative weight, from the Phase 1 error mix
OP_WEIGHTS = {'consonant': 0.60, 'mark': 0.25, 'conjunct': 0.10, 'punct_space': 0.05}


def corrupt(text, rate, rng=random):
    """Return an OCR-like noisy version of `text`. `rate` ~ expected fraction of characters edited."""
    out, chars, i = [], list(text), 0
    while i < len(chars):
        c = chars[i]
        if rng.random() >= rate:
            out.append(c); i += 1; continue
        op = rng.choices(list(OP_WEIGHTS), weights=list(OP_WEIGHTS.values()))[0]
        if op == 'consonant' and c in CONSONANT_CONFUSIONS:
            out.append(rng.choice(CONSONANT_CONFUSIONS[c]))
        elif op == 'mark' and c in MARK_CONFUSIONS:
            out.append(rng.choice(MARK_CONFUSIONS[c]))
        elif op == 'conjunct' and c in CONSONANTS:
            nxt = chars[i + 1:i + 3]
            if nxt[:1] == ['्'] and len(nxt) == 2:            # C + virama + C  -> drop the half-letter
                out.append(c); i += 3; continue
            out.append(c + '्र')                               # spurious ra-phala (seen: दोष -> द्रोष)
        elif op == 'punct_space' and c in PUNCT_CONFUSIONS:
            out.append(rng.choice(PUNCT_CONFUSIONS[c]))
        elif op == 'punct_space' and c in CONSONANTS and rng.random() < 0.3:
            out.append(c + ' ')                                # spurious word split
        else:
            out.append(c)                                      # op not applicable to this char
        i += 1
    return unicodedata.normalize('NFC', ' '.join(''.join(out).split()))


def sample_rate(rng=random):
    """Per-line noise level: 10% clean lines (teach 'leave correct text alone'), rest spread around real OCR's ~10% CER.
    Measured: effective CER ~= rate/2, so rate in [0.03, 0.45] -> CER ~1.5%-20% (real Tesseract on our pages: 9.6%)."""
    return 0.0 if rng.random() < 0.10 else rng.uniform(0.03, 0.45)


if __name__ == '__main__':
    rng = random.Random(0)
    s = 'रोगः नाम दोष वैषम्य कारणम् दोष सम्यता आरोग्य कारणम्'
    assert corrupt(s, 0.0, rng) == s
    noisy = [corrupt(s, 0.15, rng) for _ in range(200)]
    assert sum(n != s for n in noisy) > 150 and all(unicodedata.is_normalized('NFC', n) for n in noisy)
    print('ok', noisy[:3])

Writing /content/noise.py

In [ ]:
!python /content/noise.py
import sys, importlib; sys.path.insert(0, '/content'); import noise; importlib.reload(noise)

# Calibration 1: what CER does each `rate` actually produce? (not every char has an applicable op)
rng = random.Random(1)
cal = random.sample(chunks, 2000)
for r in (0.05, 0.10, 0.20, 0.30):
    print(f'rate {r:.2f} -> CER {jiwer.cer(cal, [noise.corrupt(c, r, rng) for c in cal]):.3f}')

# Calibration 2: does synthetic noise on the TEST GT look like real OCR on the TEST pages? (edit-type mix only, never trained on)
def op_mix(pairs):
    m = Counter()
    for a, b in pairs:
        for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b, autojunk=False).get_opcodes():
            if tag != 'equal': m[tag] += 1
    tot = sum(m.values()); return {k: round(v / tot, 2) for k, v in m.items()}
print('\nreal OCR edit mix :', op_mix([(flat(gt[i]), flat(ocr[BEST][i])) for i in gt]))
print('synthetic edit mix:', op_mix([(flat(gt[i]), noise.corrupt(flat(gt[i]), 0.2, rng)) for i in gt for _ in range(20)]))
print('\nexamples:')
for c in random.sample(chunks, 5):
    print(f'  CLEAN: {c}\n  NOISY: {noise.corrupt(c, 0.2, rng)}')

ok ['रोगः नाम दोष वैषम्य कारणम् दोष सम्यता आरोग्य कारणय्', 'रोगः नाम दोष तैषमय फारणभ् दोष समयता आरोग्य कारणम्', 'रोगः नाम दोष वैषम्य कारणभ् दोष समयता आरोग्य कारणम्']
rate 0.05 -> CER 0.023
rate 0.10 -> CER 0.049
rate 0.20 -> CER 0.095
rate 0.30 -> CER 0.139

real OCR edit mix : {'replace': 0.75, 'delete': 0.16, 'insert': 0.1}
synthetic edit mix: {'replace': 0.69, 'delete': 0.17, 'insert': 0.13}

examples:
  CLEAN: अन्यथा अयस्कन्तस्य शक्तिः क्षीणा भवति ।
  NOISY: अन्यथा अयस्फन्तस्य शफ्तिः क्षीणा भवति ।
  CLEAN: भवभूतिकालिदासौ समकालीनौ आस्ताम् इत्येतं
  NOISY: भवभूनीकालिदसौ ममकालीनौ आस्ताम् इत्येतं
  CLEAN: कालिया लखनऊविश्वविद्यालयस्य व्याख्याता नियुक्तः ।
  NOISY: कालिग लखनऊविश्वविदालयस्य व्याख्याता नियुक्नः ।
  CLEAN: राममोहनेन स्पष्टतया उक्तं
  NOISY: राममोहनेन सण्टतया उक्तं
  CLEAN: कार्यमेव लाभदायकं भवति ।
  NOISY: फार्यमेव लाभदायकंभवति ।

**2c. Render-then-OCR pairs (real OCR errors, no hand labelling).** Draw clean chunks as degraded line images: several Devanagari fonts, paper texture, blur, rotation, JPEG artefacts. Run the *same* Tesseract config over them and keep (OCR output → clean text). These pairs contain real Tesseract error patterns that rules can't fully capture.

Requires PIL with `raqm` (complex text layout). Without it, PIL draws Devanagari conjuncts and matras incorrectly and the images would be garbage.

In [ ]:
!apt-get -qq install -y fonts-noto-core fonts-lohit-deva > /dev/null
FONT_DIR = '/content/fonts'; os.makedirs(FONT_DIR, exist_ok=True)
for path in ['kalam/Kalam-Regular.ttf', 'tirodevanagarisanskrit/TiroDevanagariSanskrit-Regular.ttf', 'yatraone/YatraOne-Regular.ttf']:
    !wget -q -nc -P {FONT_DIR} https://github.com/google/fonts/raw/main/ofl/{path}

from PIL import ImageDraw, ImageFont, ImageFilter, features
assert features.check('raqm'), 'PIL lacks raqm -> Devanagari shaping broken'
FONTS = [f'{FONT_DIR}/{f}' for f in os.listdir(FONT_DIR)] + [
    '/usr/share/fonts/truetype/noto/NotoSansDevanagari-Regular.ttf',
    '/usr/share/fonts/truetype/noto/NotoSerifDevanagari-Regular.ttf',
    '/usr/share/fonts/truetype/lohit-devanagari/Lohit-Devanagari.ttf']
FONTS = [f for f in FONTS if os.path.exists(f)]
print(len(FONTS), 'fonts:', [os.path.basename(f) for f in FONTS])

6 fonts: ['YatraOne-Regular.ttf', 'TiroDevanagariSanskrit-Regular.ttf', 'Kalam-Regular.ttf', 'NotoSansDevanagari-Regular.ttf', 'NotoSerifDevanagari-Regular.ttf', 'Lohit-Devanagari.ttf']

In [ ]:
import io, time, numpy as np
from multiprocessing import Pool

# Tesseract is near-perfect on clean print fonts (CER 0-4%) but not on handwriting-style Kalam (~13%), whose style is close
# to the test pages. Half the renders use Kalam so the pairs contain real errors. Documented as a domain-matching choice.
FONT_W = [len(FONTS) - 1 if 'Kalam' in f else 1 for f in FONTS]

def render(text, rng):
    """Clean text -> degraded scan-like line image."""
    font = ImageFont.truetype(rng.choices(FONTS, FONT_W)[0], rng.randint(26, 52), layout_engine=ImageFont.Layout.RAQM)
    l, t, r, b = font.getbbox(text)
    pad = rng.randint(10, 30)
    W, H = r - l + 2 * pad, b - t + 2 * pad
    bg = rng.randint(175, 245)
    paper = np.clip(np.random.default_rng(rng.randint(0, 1 << 30)).normal(bg, rng.uniform(3, 22), (H, W)), 0, 255)
    img = Image.fromarray(paper.astype('uint8'))
    ImageDraw.Draw(img).text((pad - l, pad - t), text, font=font, fill=rng.randint(0, 80))
    img = img.rotate(rng.uniform(-2, 2), expand=True, fillcolor=bg)
    if rng.random() < 0.8: img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 1.6)))
    if rng.random() < 0.6:                                    # low-res scan: downsample then upsample
        s = rng.uniform(0.35, 0.8); img = img.resize((int(img.width * s), int(img.height * s))).resize(img.size)
    buf = io.BytesIO(); img.save(buf, 'JPEG', quality=rng.randint(20, 90))
    return Image.open(buf), os.path.basename(font.path)

def render_ocr(args):
    idx, text = args
    rng = random.Random(idx)
    img, font = render(text, rng)
    return {'clean': text, 'noisy': norm(pytesseract.image_to_string(img, lang='san', config='--psm 7')).replace('\n', ' '), 'font': font}

t0 = time.time()
with Pool(os.cpu_count()) as p:
    trial = p.map(render_ocr, enumerate(random.Random(7).sample(chunks, 60)))
print(f'{(time.time() - t0) / 60 * 1000:.0f} ms/line on {os.cpu_count()} CPUs')
print(f"overall CER {jiwer.cer([x['clean'] for x in trial], [x['noisy'] or '-' for x in trial]):.3f}")
by_font = {}
for x in trial: by_font.setdefault(x['font'], []).append(x)
for f, xs in by_font.items():
    print(f"{f:40} CER {jiwer.cer([x['clean'] for x in xs], [x['noisy'] or '-' for x in xs]):.3f}  n={len(xs)}")
for x in trial[:4]: print(f"  CLEAN: {x['clean']}\n  OCR  : {x['noisy']}  [{x['font']}]")

158 ms/line on 2 CPUs
overall CER 0.100
NotoSerifDevanagari-Regular.ttf          CER 0.041  n=2
TiroDevanagariSanskrit-Regular.ttf       CER 0.019  n=5
Lohit-Devanagari.ttf                     CER 0.027  n=8
Kalam-Regular.ttf                        CER 0.150  n=36
NotoSansDevanagari-Regular.ttf           CER 0.021  n=3
YatraOne-Regular.ttf                     CER 0.048  n=6
  CLEAN: तत् केवलम् अस्य सन्निधानात् प्रकृतौ एव
  OCR  : तत् केवलम् अस्य सन्निधानात् प्रकृतौ एव  [NotoSerifDevanagari-Regular.ttf]
  CLEAN: इन्द्र-अग्नि-वरुण-मरुत्-सोम-अश्विनौ उपास्यदेवेषु मुख्याः।
  OCR  : इन्द्र-अग्नि-वरुण-मरुत्-सोम-अश्िनौ उपास्यदेवेषु मुख्याः।  [TiroDevanagariSanskrit-Regular.ttf]
  CLEAN: त्यक्तपुत्रवित्तलोकैषणः संन्यासी आत्माराम आत्मक्रीडः
  OCR  : त्यक्तपुत्रवित्तलोकैषणः संन्यासी आत्माराम आत्मक्रीडः  [Lohit-Devanagari.ttf]
  CLEAN: वर्षेषु व्यतितेषु ' पाण्डवा अभूवन् ।
  OCR  : ठर्षेषु व्मतितेषु ' पाण्डवा अभूठन् ।  [Kalam-Regular.ttf]

In [ ]:
# Disjoint pools from the shuffled corpus: no clean line appears in more than one split.
pool = chunks[:]; random.Random(42).shuffle(pool)
N_RENDER, N_RULE = 6000, 60000
render_pool = pool[:N_RENDER]
rule_pool = pool[N_RENDER:N_RENDER + N_RULE]

# Render+OCR in resumable batches (~158 ms/line on 2 CPUs). Re-run this cell until it prints "done".
RENDER_OUT, BATCH = f'{WORK}/render_pairs.jsonl', 1500
done = sum(1 for _ in open(RENDER_OUT, encoding='utf-8')) if os.path.exists(RENDER_OUT) else 0
todo = list(enumerate(render_pool))[done:done + BATCH]
if todo:
    t0 = time.time()
    with Pool(os.cpu_count()) as p, open(RENDER_OUT, 'a', encoding='utf-8') as f:
        for rec in p.imap(render_ocr, todo, chunksize=16):   # imap keeps order -> resume index stays valid
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    print(f'rendered {done + len(todo)}/{N_RENDER} ({time.time() - t0:.0f}s this batch)')
else:
    print('done:', done)

rendered 1500/6000 (206s this batch)
rendered 3000/6000 (197s this batch)
rendered 4500/6000 (196s this batch)
rendered 6000/6000 (196s this batch)

**2d. Final splits.** Training data is rule-noise pairs plus render-OCR pairs. The render-OCR pairs are counted twice because they're closer to real errors. Dev and test hold out both kinds separately, so Phase 4 can report each one. The real 5-page set stays separate as the main test.

In [ ]:
rrng = random.Random(123)
rule = [{'clean': c, 'noisy': noise.corrupt(c, noise.sample_rate(rrng), rrng), 'src': 'rule'} for c in rule_pool]
rend = [dict(json.loads(l), src='render') for l in open(RENDER_OUT, encoding='utf-8')]
rend = [r for r in rend if r['noisy']]                        # drop lines where OCR returned nothing
assert len(rend) > 0.95 * N_RENDER, f'render step incomplete: {len(rend)}'

splits = {
    'train': rule[:-2000] + rend[:-1000] * 2,               # render pairs upweighted x2 (closer to real OCR errors)
    'dev': rule[-2000:-1000] + rend[-1000:-500],
    'test_rule': rule[-1000:],
    'test_render': rend[-500:],
}
random.Random(0).shuffle(splits['train'])
for name_, rows_ in splits.items():
    with open(f'{WORK}/{name_}.jsonl', 'w', encoding='utf-8') as f:
        for r in rows_: f.write(json.dumps(r, ensure_ascii=False) + '\n')
    cer = jiwer.cer([r['clean'] for r in rows_], [r['noisy'] for r in rows_])
    unchanged = sum(r['clean'] == r['noisy'] for r in rows_) / len(rows_)
    print(f'{name_:12} n={len(rows_):6}  do-nothing CER {cer:.3f}  already-correct lines {unchanged:.0%}')

# leakage checks: test clean lines never seen in train; real test pages never in any split
train_clean = {r['clean'] for r in splits['train']}
assert not any(r['clean'] in train_clean for k in ('test_rule', 'test_render', 'dev') for r in splits[k])
assert not any(t in r['clean'] for r in splits['train'] for t in test_trigrams)
lens = [len(r['clean'].encode()) for r in splits['train']]
print(f'\nUTF-8 bytes per target line (ByT5 sees bytes): mean {np.mean(lens):.0f}, p99 {np.percentile(lens, 99):.0f}, max {max(lens)}')

if os.path.exists('/content/drive/MyDrive'):
    !mkdir -p /content/drive/MyDrive/sanskrit_ocr && cp {WORK}/*.jsonl {WORK}/phase1.json /content/drive/MyDrive/sanskrit_ocr/
    print('backed up to Drive')

train        n= 67984  do-nothing CER 0.074  already-correct lines 24%
dev          n=  1500  do-nothing CER 0.081  already-correct lines 23%
test_rule    n=  1000  do-nothing CER 0.074  already-correct lines 23%
test_render  n=   500  do-nothing CER 0.088  already-correct lines 31%

UTF-8 bytes per target line (ByT5 sees bytes): mean 121, p99 285, max 349
backed up to Drive

### Phase 2 summary
| split | n | do-nothing CER | already-correct lines |
|---|---|---|---|
| train (rule + render×2) | 67,984 | 7.4% | 24% |
| dev | 1,500 | 8.1% | 23% |
| test_rule | 1,000 | 7.4% | 23% |
| test_render | 500 | 8.8% | 31% |
| **real pages (Phase 1)** | 5 pages | **9.6%** | – |

- **Noise is realistic:** synthetic edit mix 69/17/13 (sub/del/ins) vs real 75/16/10. Render-OCR on a Kalam-style font reproduces the exact real confusions (व→ठ, य→म).
- **~24% of inputs are already correct.** The model must learn to copy, not rewrite, which is our main guard against over-correction (hallucination).
- **Sequence length:** targets average 121 UTF-8 bytes (p99 285, max 349), so `max_length=384` drops nothing. Devanagari is 3 bytes per character in UTF-8, which is ByT5's main cost.
- **Leakage checks pass:** dev and test lines never appear in train, and no 3-word window from the real pages appears anywhere.

**Limitations of Phase 2**
- **The training text doesn't match the test text.** Wikipedia is modern encyclopaedic Sanskrit prose. The test pages are Ayurvedic shastra, and real targets would include verse, heavy sandhi and classical vocabulary. Better sources (GRETIL, DCS, Gita) were skipped for time.
- **The "clean" labels aren't perfectly clean.** Wikipedia has its own typos (e.g. मन्षुयस्तु), so the model can learn to reproduce them.
- **Rule noise treats every character independently.** Real OCR errors cluster: a smudged word gets several errors, and a whole line can go bad. The confusion table is hand-made from 51 observations plus general knowledge, so confusions specific to other fonts are missing.
- **Matching the font may inflate results.** Making half the renders Kalam helps the real pages because their font looks similar. A different font could do noticeably worse. Phase 4 reports test_render by font to show this.
- **Rendered lines are only an approximation of a scan.** There's no ink bleed, broken strokes, curved baselines or physical page damage. Lines where OCR returned nothing were dropped, so complete OCR failure isn't covered.
- **The scope is single lines.** No page layout, line-splitting errors or words broken across lines, and no Roman-transliterated (IAST) text, even though the brief mentions transliteration inconsistencies.

## Phase 3: Model and training
**3a. Why ByT5? A tokenizer check.** OCR correction is a character-level edit task. A subword tokenizer turns a one-character OCR error into a *completely different* token sequence. It can also map rare conjuncts to `<unk>` or quietly rewrite Unicode. We compare `byt5-small` against `mt5-small`, the obvious alternative, on our training text.

In [ ]:
from transformers import AutoTokenizer
toks = {m: AutoTokenizer.from_pretrained(m) for m in ('google/byt5-small', 'google/mt5-small')}
sample = [json.loads(l) for l in open(f'{WORK}/dev.jsonl', encoding='utf-8')][:1000]

print(f"{'tokenizer':18} {'tok/word':>8} {'tok/char':>8} {'unk%':>6} {'roundtrip-exact':>15} {'1-char edit -> tokens changed':>30}")
for m, tk in toks.items():
    ids = [tk(r['clean'], add_special_tokens=False)['input_ids'] for r in sample]
    n_tok = sum(map(len, ids)); n_word = sum(len(r['clean'].split()) for r in sample); n_char = sum(len(r['clean']) for r in sample)
    unk = sum(i == tk.unk_token_id for x in ids for i in x) / n_tok if tk.unk_token_id is not None else 0
    exact = np.mean([tk.decode(x) == r['clean'] for x, r in zip(ids, sample)])
    # how many tokens differ between clean and its noisy version (only pairs differing by exactly one edit)
    diffs = []
    for r in sample:
        ops_ = [o for o in difflib.SequenceMatcher(None, r['clean'], r['noisy'], autojunk=False).get_opcodes() if o[0] != 'equal']
        if len(ops_) == 1:
            a, b = tk.tokenize(r['clean']), tk.tokenize(r['noisy'])
            diffs.append(sum(o[0] != 'equal' and max(o[2] - o[1], o[4] - o[3]) for o in difflib.SequenceMatcher(None, a, b).get_opcodes()))
    print(f"{m.split('/')[1]:18} {n_tok / n_word:8.2f} {n_tok / n_char:8.2f} {unk:6.2%} {exact:15.1%} {np.mean(diffs):30.2f}")

w = 'संयोगः'
for m, tk in toks.items(): print(f'{m:20} {w} -> {tk.tokenize(w)}')

tokenizer          tok/word tok/char   unk% roundtrip-exact  1-char edit -> tokens changed
byt5-small            23.49     2.79  0.00%          100.0%                           2.11
mt5-small              3.73     0.44  0.00%           99.9%                           2.06
google/byt5-small    संयोगः -> ['à', '¤', '¸', 'à', '¤', '\x82', 'à', '¤', '¯', 'à', '¥', '\x8b', 'à', '¤', '\x97', 'à', '¤', '\x83']
google/mt5-small     संयोगः -> ['▁सं', 'योग', 'ः']

**Tokenizer finding:** mT5's SentencePiece vocabulary handles Devanagari Sanskrit **better than expected**. It produces 0% `<unk>`, 99.9% exact round-trip, and splits words sensibly (संयोगः → `▁सं|योग|ः`). A one-character OCR error changes about 2 tokens for *both* models, so "subword fragmentation breaks edits" isn't a strong argument here.

**We still choose ByT5-small, for these reasons:**
1. **Nothing is ever lost or unknown, by design.** Real OCR output contains Latin look-alikes (`|`, `l`, `:`), stray marks and invalid matra sequences. A byte model can read and write *any* string. A fixed vocabulary only *happens* to cover our sample (99.9% ≠ 100%).
2. **Corrections act directly on characters.** A matra fix is a 1–3 byte edit, which suits the copy-mostly nature of the task.
3. **Published work:** the ByT5 paper reports byte models are clearly more robust than mT5 on noisy and misspelled input.

**What it costs:** ByT5 sequences are **6.3× longer** (2.79 tokens per character vs 0.44). Training and generation are much slower, which is the price on a T4. **mT5-small is a cheaper alternative worth testing** and is listed under future work. We kept the tokenizers unchanged: adding vocabulary to mT5 isn't needed (0% unknown), and ByT5 has nothing to adapt.

**3b. Training (`train.py`).** Full fine-tuning of ByT5-small (300M parameters; small enough that LoRA isn't needed).
- **Precision:** fp32 on the T4. It has no native bf16, and T5-family models overflow to NaN in fp16. The script switches to bf16 automatically on an L4 or A100.
- **Optimizer and batching:** Adafactor (low optimizer memory), effective batch 32 (8 × 4 gradient accumulation; batch 16 ran out of memory in fp32), linear LR schedule with 3% warmup. Length-grouped batching cuts padding, which matters because sequences are bytes.
- **Checkpoints and model selection:** checkpoints go to **Google Drive** every 400 steps and training resumes automatically after a runtime reset. The best checkpoint is picked by dev CER from actual generated output, not by loss.
- **Length of run:** 1 epoch (2,125 steps), chosen after the smoke run measured 1.85 s/step. 2 epochs would take about 2.2 h on a T4.

A short smoke run comes first to measure speed and check the pipeline end to end.

In [ ]:
%%writefile /content/train.py
"""Fine-tune ByT5 for Sanskrit post-OCR correction: noisy OCR line -> clean line.

Usage: python train.py --data /content/work --out /content/drive/MyDrive/sanskrit_ocr/byt5
Expects {data}/train.jsonl and {data}/dev.jsonl with fields `noisy`, `clean`.
"""
import argparse
import glob
import os

import jiwer
import numpy as np
import torch
from datasets import load_dataset
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments)

p = argparse.ArgumentParser()
p.add_argument('--data', default='/content/work')
p.add_argument('--out', default='/content/drive/MyDrive/sanskrit_ocr/byt5')
p.add_argument('--model', default='google/byt5-small')
p.add_argument('--epochs', type=float, default=1)        # 1 epoch ~= 2125 steps ~= 70 min on a T4 (1.85 s/step)
p.add_argument('--max_steps', type=int, default=-1)
p.add_argument('--bs', type=int, default=8)          # bs16 OOMs on a T4 in fp32 at 384 bytes
p.add_argument('--accum', type=int, default=4)       # effective batch 32
p.add_argument('--lr', type=float, default=5e-4)
p.add_argument('--max_len', type=int, default=384)        # p99 target = 285 bytes, max 349
p.add_argument('--eval_steps', type=int, default=500)
p.add_argument('--eval_n', type=int, default=300)         # dev lines decoded per eval (generation is slow for bytes)
args = p.parse_args()

tok = AutoTokenizer.from_pretrained(args.model)
model = AutoModelForSeq2SeqLM.from_pretrained(args.model)

ds = load_dataset('json', data_files={'train': f'{args.data}/train.jsonl', 'dev': f'{args.data}/dev.jsonl'})


def prep(b):
    enc = tok(b['noisy'], max_length=args.max_len, truncation=True)
    enc['labels'] = tok(text_target=b['clean'], max_length=args.max_len, truncation=True)['input_ids']
    return enc


ds = ds.map(prep, batched=True, remove_columns=ds['train'].column_names)
dev = ds['dev'].shuffle(seed=0).select(range(min(args.eval_n, len(ds['dev']))))


def compute_metrics(ev):
    preds, labels = ev
    preds = np.where(preds < 0, tok.pad_token_id, preds)
    labels = np.where(labels < 0, tok.pad_token_id, labels)
    hyp = tok.batch_decode(preds, skip_special_tokens=True)
    ref = tok.batch_decode(labels, skip_special_tokens=True)
    return {'cer': jiwer.cer(ref, hyp), 'wer': jiwer.wer(ref, hyp)}


# T4 (sm_75) has no native bf16, and T5-family models overflow to NaN in fp16 -> fp32 there; bf16 on L4/A100.
bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8

targs = Seq2SeqTrainingArguments(
    output_dir=args.out,
    per_device_train_batch_size=args.bs,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=args.accum,
    learning_rate=args.lr,
    optim='adafactor',                       # T5's native optimizer; far less optimizer memory than AdamW
    lr_scheduler_type='linear',
    warmup_steps=0.03,                       # transformers>=5: float in [0,1) = ratio of total steps
    num_train_epochs=args.epochs,
    max_steps=args.max_steps,
    bf16=bf16,
    fp16=False,
    train_sampling_strategy='group_by_length',  # batch similar lengths -> less padding (bytes make padding expensive)
    eval_strategy='steps',
    eval_steps=args.eval_steps,
    save_steps=args.eval_steps,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=args.max_len,
    logging_steps=50,
    dataloader_num_workers=2,
    report_to='none',
)

trainer = Seq2SeqTrainer(model=model, args=targs, train_dataset=ds['train'], eval_dataset=dev,
                         data_collator=DataCollatorForSeq2Seq(tok, model=model), processing_class=tok,
                         compute_metrics=compute_metrics)

# Resume if a checkpoint exists (Colab runtimes get recycled; checkpoints live on Drive).
last = sorted(glob.glob(f'{args.out}/checkpoint-*'), key=lambda d: int(d.rsplit('-', 1)[1]))
trainer.train(resume_from_checkpoint=last[-1] if last else None)
trainer.save_model(f'{args.out}/final')
tok.save_pretrained(f'{args.out}/final')
print('saved', f'{args.out}/final', '| best dev CER:', trainer.state.best_metric)

Overwriting /content/train.py

In [ ]:
# Smoke run: 40 steps, tiny eval. Checks the pipeline end to end and measures step time before committing an hour.
# bs16 OOMs in fp32 on the longest (first, due to group_by_length) batch -> bs8 x accum4 = same effective batch 32.
!rm -rf /content/smoke && cd /content && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train.py --out /content/smoke --bs 8 --accum 4 --max_steps 40 --eval_steps 20 --eval_n 32 2>&1 | grep -vE "Warning|warn|Map|Generating|Loading|^\s*$" | tr '\r' '\n' | grep -E "it/s|s/it|loss|cer|Error|saved" | tail -15

100%|██████████| 40/40 [02:40<00:00,  1.85s/it]
{'eval_loss': '0.2069', 'eval_cer': '0.1374', 'eval_wer': '0.3494', 'eval_runtime': '4.372', 'eval_samples_per_second': '7.32', 'eval_steps_per_second': '0.229', 'epoch': '0.01883'}
Writing model shards: 100%|██████████| 1/1 [01:03<00:00, 63.31s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].
{'train_runtime': '228.7', 'train_samples_per_second': '5.596', 'train_steps_per_second': '0.175', 'train_loss': '2.26', 'epoch': '0.01883'}
Writing model shards: 100%|██████████| 1/1 [00:45<00:00, 45.53s/it]
saved /content/smoke/final | best dev CER: 0.1373873873873874

**3c. Full training run.** Training runs as a **fully detached process** (`subprocess.Popen(..., start_new_session=True)`): the cell returns immediately and the notebook stays usable for monitoring.

*What went wrong the first time (kept as an engineering note):* the first launch used `!nohup python train.py ... &`. In Colab the shell cell stayed attached to the child process, so the notebook was blocked for the whole run. When the browser session reconnected, training stalled after `checkpoint-400`. Because checkpoints were on Drive, nothing was lost. The cell below restores the data splits from the Drive backup, downloads `train.py` from the repo and **resumes from the latest checkpoint** (optimizer, LR schedule and data position included). It is safe to re-run after any runtime reset.

In [ ]:
# Resume training from the latest Drive checkpoint. Self-contained; safe to re-run.
import os, subprocess, glob, shutil, time
D, WORK = '/content/drive/MyDrive/sanskrit_ocr', '/content/work'
os.makedirs(WORK, exist_ok=True)
for f in ['train.jsonl', 'dev.jsonl', 'test_rule.jsonl', 'test_render.jsonl', 'phase1.json']:
    if not os.path.exists(f'{WORK}/{f}'):
        shutil.copy(f'{D}/{f}', WORK)                       # restore data from the Drive backup
subprocess.run('pip -q install jiwer pytesseract', shell=True)
subprocess.run('wget -q -O /content/train.py https://raw.githubusercontent.com/Mrunal31-Stark/OCR_Extraction_indic_text/main/src/train.py',
               shell=True, check=True)
print('resuming from:', [os.path.basename(c) for c in sorted(glob.glob(f'{D}/byt5/checkpoint-*'))])

# Fully detached process: this cell returns; the notebook stays usable while training runs.
LOG = f'{WORK}/train.log'
proc = subprocess.Popen(
    ['python', '/content/train.py', '--data', WORK, '--out', f'{D}/byt5', '--eval_steps', '400'],
    stdout=open(LOG, 'a'), stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True,
    env={**os.environ, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}, cwd='/content')
time.sleep(100)
print('STATUS:', 'running' if proc.poll() is None else f'EXITED with code {proc.returncode}')
tail = open(LOG, encoding='utf-8', errors='ignore').read().replace('\r', '\n').splitlines()
print('\n'.join(l for l in tail if l.strip() and 'Map' not in l)[-1000:])

resuming from: ['checkpoint-400']
STATUS: running
Loading weights: 100%|██████████| 172/172 [00:00<00:00, 20056.17it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Generating train split: 67984 examples [00:00, 559025.64 examples/s]
Generating dev split: 1500 examples [00:00, 339858.25 examples/s]
[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].

In [ ]:
# Monitor (safe to re-run any time): process alive?, latest progress bar, every logged loss / dev CER.
import re, ast, subprocess
log = open('/content/work/train.log', encoding='utf-8', errors='ignore').read().replace('\r', '\n')
alive = subprocess.run("ps aux | grep '[t]rain.py' | wc -l", shell=True, capture_output=True, text=True).stdout.strip()
print('process alive:', alive != '0')
bars = re.findall(r'(\d+)/2125 \[([^\]]+)\]', log)
print('progress: step %s/2125  [%s]' % bars[-1] if bars else 'progress: no bar yet')
for m in re.finditer(r"\{'(?:loss|eval_loss)'[^}]*\}", log):
    r = ast.literal_eval(m.group(0))
    print(f"  step~{float(r['epoch']) * 2125:5.0f}  " + (f"dev CER {float(r['eval_cer']):.4f}  WER {float(r['eval_wer']):.4f}" if 'eval_cer' in r else f"loss {r['loss']}"))
errs = [l for l in log.splitlines() if 'Error' in l or 'Traceback' in l]
print('errors:', errs[-3:] if errs else 'none')

process alive: True
progress: step 1768/2125  [1:09:21<20:35,  3.46s/it]
  step~  450  loss 0.3259
  step~  500  loss 0.3198
  step~  550  loss 0.2945
  step~  600  loss 0.3211
  step~  650  loss 0.2907
  step~  700  loss 0.2828
  step~  750  loss 0.27
  step~  800  loss 0.2661
  step~  800  dev CER 0.0698  WER 0.2965
  step~  850  loss 0.2615
  step~  900  loss 0.2584
  step~  950  loss 0.2655
  step~ 1000  loss 0.2533
  step~ 1050  loss 0.246
  step~ 1100  loss 0.2527
  step~ 1150  loss 0.2393
  step~ 1200  loss 0.2339
  step~ 1200  dev CER 0.0569  WER 0.2625
  step~ 1250  loss 0.2254
  step~ 1300  loss 0.2236
  step~ 1350  loss 0.2262
  step~ 1400  loss 0.2309
  step~ 1450  loss 0.2257
  step~ 1500  loss 0.2135
  step~ 1550  loss 0.2245
  step~ 1600  loss 0.204
  step~ 1600  dev CER 0.0524  WER 0.2461
errors: none

In [ ]:
import subprocess
print(subprocess.run('nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader', shell=True, capture_output=True, text=True).stdout)

97 %, 8611 MiB

**3d. Early stop because of Colab GPU disconnects.** The Google Colab GPU runtime kept disconnecting during the run, so I stopped training at step 1,768 of 2,125. The best saved checkpoint by dev CER (**`checkpoint-1600`, dev CER 5.2%**) is the final model. The dev curve (7.8% → 7.0% → 5.7% → 5.2%) was still going down, so finishing the epoch (about 30 min more on a T4, by re-running the resume cell) would probably help a little; see the report.

> ⚠️ **Don't run the next cell during a normal reproduction.** It kills the training process. Skip it and let training finish; the final model is then saved to `MyDrive/sanskrit_ocr/byt5/final`, which Phase 4 picks up automatically.

In [ ]:
# Early stop (Colab kept disconnecting): stop training and use the best checkpoint so far (by dev CER).
import subprocess, glob, json, os, time
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
print(sh("ps -eo pid,cmd | grep '[t]rain.py'"))
sh("pkill -9 -f train.py"); time.sleep(5)
print('train.py still running:', sh("ps aux | grep '[t]rain.py' | wc -l"), '| GPU mem:', sh('nvidia-smi --query-gpu=memory.used --format=csv,noheader'))
D = '/content/drive/MyDrive/sanskrit_ocr'
for c in sorted(glob.glob(f'{D}/byt5/checkpoint-*'), key=lambda d: int(d.rsplit('-', 1)[1])):
    st = json.load(open(f'{c}/trainer_state.json'))
    print(os.path.basename(c), '| model.safetensors:', os.path.exists(f'{c}/model.safetensors'), '| best so far:', os.path.basename(st['best_model_checkpoint']), round(st['best_metric'], 4))

6118 python3 /content/train.py --data /content/work --out /content/drive/MyDrive/sanskrit_ocr/byt5 --eval_steps 400
   6612 python3 /content/train.py --data /content/work --out /content/drive/MyDrive/sanskrit_ocr/byt5 --eval_steps 400
   6613 python3 /content/train.py --data /content/work --out /content/drive/MyDrive/sanskrit_ocr/byt5 --eval_steps 400
train.py still running: 0 | GPU mem: 0 MiB
checkpoint-1200 | model.safetensors: True | best so far: checkpoint-1200 0.0569
checkpoint-1600 | model.safetensors: True | best so far: checkpoint-1600 0.0524

## Phase 4: Evaluation
Loads the best checkpoint (selected by dev CER) and evaluates on:
1. **The 5 real page images:** Tesseract → corrector, page by page (the result that matters most).
2. **Noisy images we created:** the 500 held-out render+OCR lines (by font), plus 6 fresh degraded line images saved as PNGs for the report.
3. **Synthetic rule noise:** 1,000 held-out lines.

Metrics: CER/WER before → after, fully-fixed rate, **over-correction rate** (correct lines the model broke, i.e. hallucination), made-worse rate, error-type breakdown, bootstrap 95% CI on the real pages, greedy vs beam search.

In [ ]:
# 4a. Setup: OCR + fonts on this runtime, repo code, best checkpoint.
import os, sys, re, json, glob, random, unicodedata, subprocess
subprocess.run('apt-get -qq install -y tesseract-ocr tesseract-ocr-san fonts-noto-core fonts-lohit-deva > /dev/null', shell=True)
subprocess.run('pip -q install jiwer pytesseract', shell=True)
REPO = '/content/OCR_Extraction_indic_text'
if not os.path.exists(REPO):
    subprocess.run(f'git clone -q https://github.com/Mrunal31-Stark/OCR_Extraction_indic_text.git {REPO}', shell=True, check=True)
subprocess.run(f'git -C {REPO} pull -q', shell=True)
sys.path.insert(0, f'{REPO}/src')
import jiwer, pytesseract, numpy as np
from PIL import Image
from infer import Corrector
from evaluate import full_report, bootstrap_cer, error_types

D, WORK = '/content/drive/MyDrive/sanskrit_ocr', '/content/work'
state = json.load(open(sorted(glob.glob(f'{D}/byt5/checkpoint-*/trainer_state.json'), key=os.path.getmtime)[-1]))
print('dev CER per eval:', [(h['step'], round(h['eval_cer'], 4)) for h in state['log_history'] if 'eval_cer' in h])
MODEL = f'{D}/byt5/final' if os.path.exists(f'{D}/byt5/final') else state['best_model_checkpoint']
print('using model:', MODEL)
corr = Corrector(MODEL)

def norm(s):
    s = unicodedata.normalize('NFC', s).replace('‌', '').replace('‍', '')
    s = re.sub(r'[ \t]+', ' ', s)
    return '\n'.join(l.strip() for l in s.strip().splitlines() if l.strip())
flat = lambda s: ' '.join(s.split())

# sanity: an already-correct line must come back unchanged, and reloading gives identical outputs (tied-embedding warning check)
probe = ['तस्य हितं च अहितं च मानम् च', 'तस्त्र हितं च अदितं च']
out1 = corr.correct(probe); out2 = Corrector(MODEL).correct(probe)
print(out1); assert out1 == out2, 'reloaded model gives different outputs'

dev CER per eval: [(400, 0.0782), (800, 0.0698), (1200, 0.0569), (1600, 0.0524)]
using model: /content/drive/MyDrive/sanskrit_ocr/byt5/checkpoint-1600
['तस्य हितं च अहितं च मानम् च', 'तस्य हितं च अदितं च']

In [ ]:
# 4b. REAL page images: Tesseract (same config as Phase 1 baseline) -> corrector, line by line.
DATA = f'{REPO}/Data'
pages, line_pairs = {}, []          # line_pairs: (gt_line, ocr_line, corrected_line) for bootstrap CIs
for i in range(1, 6):
    gt = norm(open(f'{DATA}/page_{i}.md', encoding='utf-8').read())
    ocr = norm(pytesseract.image_to_string(Image.open(f'{DATA}/page_{i}.png'), lang='san', config='--psm 6'))
    fixed = corr.correct(ocr.splitlines())
    pages[i] = {'gt': gt, 'ocr': ocr, 'fixed': '\n'.join(fixed)}
    g_lines, o_lines = gt.splitlines(), ocr.splitlines()
    if len(g_lines) == len(o_lines):                                  # OCR kept the line structure -> line-level pairs
        line_pairs += list(zip(g_lines, o_lines, fixed))
    else:
        line_pairs.append((flat(gt), flat(ocr), flat('\n'.join(fixed))))
    print(f"page {i}: CER {jiwer.cer(flat(gt), flat(ocr)):.3f} -> {jiwer.cer(flat(gt), flat(pages[i]['fixed'])):.3f} | "
          f"WER {jiwer.wer(flat(gt), flat(ocr)):.3f} -> {jiwer.wer(flat(gt), flat(pages[i]['fixed'])):.3f}")

G, O, F = ([flat(p[k]) for p in pages.values()] for k in ('gt', 'ocr', 'fixed'))
real = full_report(G, O, F)
lo_b, hi_b = bootstrap_cer([p[0] for p in line_pairs], [p[1] for p in line_pairs])
lo_a, hi_a = bootstrap_cer([p[0] for p in line_pairs], [p[2] for p in line_pairs])
print(f"\nALL 5 PAGES  CER {real['cer_before']:.3f} [95% CI {lo_b:.3f}-{hi_b:.3f}]  ->  {real['cer_after']:.3f} [95% CI {lo_a:.3f}-{hi_a:.3f}]")
print(f"             WER {real['wer_before']:.3f} -> {real['wer_after']:.3f}   ({len(line_pairs)} line pairs)")
print('error types before:', real['types_before'], '\nerror types after :', real['types_after'])
for i, p in pages.items():
    print(f"\n--- page {i} ---\nGT   : {flat(p['gt'])}\nOCR  : {flat(p['ocr'])}\nFIXED: {flat(p['fixed'])}")

page 1: CER 0.121 -> 0.066 | WER 0.343 -> 0.257
page 2: CER 0.058 -> 0.074 | WER 0.200 -> 0.300
page 3: CER 0.098 -> 0.107 | WER 0.368 -> 0.368
page 4: CER 0.097 -> 0.090 | WER 0.400 -> 0.350
page 5: CER 0.092 -> 0.085 | WER 0.381 -> 0.381

ALL 5 PAGES  CER 0.096 [95% CI 0.074-0.125]  ->  0.082 [95% CI 0.064-0.109]
             WER 0.339 -> 0.322   (20 line pairs)
error types before: {'consonant': 26, 'conjunct': 12, 'mark': 13}
error types after : {'consonant': 23, 'mark': 15, 'conjunct': 8}

--- page 1 ---
GT   : अथ आयुर्वेदः नाम शास्त्रम् आयुर्वेदः नाम आयुः वेदयति इति आयुर्वेदः आयुः नाम शरीर इन्द्रिय सत्त्व आत्म संयोगः तस्य हिताहितं सुखं दुःखम् आयुः तस्य हितं च अहितं च मानम् च तच् च यत्र उक्तं तत् आयुर्वेदः
OCR  : अथ आयुर्वेदः नाम शास्त्रम् आयुर्वेदः नाम आगुः ठेट्रमति इति आयुर्वेद आरूः नाम शरीर इन्दिम सत्त्व आत्म संयोगः तस्त्र हिताहितं युखं दृखम् आयुः तस्त्र हितं च अदितं च मानम् च तच् च मत्र उक्तं तत् आयुर्वेद
FIXED: अथ आयुर्वेदः नाम शास्त्रम् आयुर्वेदः नाम आगुः वेटयति इति आयुर्वेद 

In [ ]:
# 4c. Held-out test sets: rendered+OCR images (by font) and rule noise.
load = lambda n: [json.loads(l) for l in open(f'{WORK}/{n}.jsonl', encoding='utf-8')]
results = {'real_pages': real}
for name in ('test_render', 'test_rule'):
    rows = load(name)
    pred = corr.correct([r['noisy'] for r in rows], batch_size=48)
    for r, p in zip(rows, pred): r['pred'] = p
    rep = full_report([r['clean'] for r in rows], [r['noisy'] for r in rows], pred)
    results[name] = rep
    print(f"{name:12} CER {rep['cer_before']:.3f} -> {rep['cer_after']:.3f} | WER {rep['wer_before']:.3f} -> {rep['wer_after']:.3f} | "
          f"fully fixed {rep['fully_fixed']:.0%} | over-correction {rep['over_correction']:.1%} | made worse {rep['made_worse']:.1%}")
    print(f"{'':12} error types {rep['types_before']} -> {rep['types_after']}")
    if name == 'test_render':
        by_font = {}
        for r in rows: by_font.setdefault(r['font'], []).append(r)
        for f, rs in sorted(by_font.items()):
            c = [r['clean'] for r in rs]
            print(f"   {f:38} n={len(rs):3}  CER {jiwer.cer(c, [r['noisy'] for r in rs]):.3f} -> {jiwer.cer(c, [r['pred'] for r in rs]):.3f}")
        results['test_render_by_font'] = {f: [len(rs), jiwer.cer([r['clean'] for r in rs], [r['noisy'] for r in rs]),
                                              jiwer.cer([r['clean'] for r in rs], [r['pred'] for r in rs])] for f, rs in by_font.items()}
    globals()[f'rows_{name}'] = rows

# Decoding: greedy vs 4-beam on the real pages + render set
for beams in (4,):
    F4 = [flat('\n'.join(corr.correct(p['ocr'].splitlines(), num_beams=beams))) for p in pages.values()]
    r4 = rows_test_render; p4 = corr.correct([r['noisy'] for r in r4], num_beams=beams)
    results[f'beam{beams}'] = {'real_cer': jiwer.cer(G, F4), 'render_cer': jiwer.cer([r['clean'] for r in r4], p4)}
    print(f"\nbeam={beams}: real pages CER {results[f'beam{beams}']['real_cer']:.3f} (greedy {real['cer_after']:.3f}) | "
          f"render CER {results[f'beam{beams}']['render_cer']:.3f} (greedy {results['test_render']['cer_after']:.3f})")
json.dump({'results': results, 'pages': pages}, open(f'{D}/phase4_results.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

test_render  CER 0.088 -> 0.074 | WER 0.326 -> 0.278 | fully fixed 7% | over-correction 7.1% | made worse 13.6%
             error types {'consonant': 616, 'conjunct': 329, 'mark': 312, 'space/punct': 84} -> {'conjunct': 261, 'mark': 304, 'consonant': 435, 'space/punct': 63}
   Kalam-Regular.ttf                      n=241  CER 0.152 -> 0.122
   Lohit-Devanagari.ttf                   n= 49  CER 0.021 -> 0.022
   NotoSansDevanagari-Regular.ttf         n= 52  CER 0.019 -> 0.019
   NotoSerifDevanagari-Regular.ttf        n= 58  CER 0.019 -> 0.021
   TiroDevanagariSanskrit-Regular.ttf     n= 55  CER 0.036 -> 0.040
   YatraOne-Regular.ttf                   n= 45  CER 0.029 -> 0.031
test_rule    CER 0.074 -> 0.034 | WER 0.360 -> 0.185 | fully fixed 30% | over-correction 13.7% | made worse 6.8%
             error types {'consonant': 1495, 'conjunct': 645, 'mark': 351, 'space/punct': 128} -> {'mark': 313, 'consonant': 540, 'conjunct': 270, 'space/punct': 69}

beam=4: real pages CER 0.077 (greedy

In [ ]:
# 4d. Fresh noisy IMAGES for the report: render held-out test sentences as degraded scans -> Tesseract -> corrector.
from PIL import ImageDraw, ImageFont, ImageFilter
import io
FONT_DIR = '/content/fonts'; os.makedirs(FONT_DIR, exist_ok=True)
for path in ['kalam/Kalam-Regular.ttf', 'tirodevanagarisanskrit/TiroDevanagariSanskrit-Regular.ttf']:
    subprocess.run(f'wget -q -nc -P {FONT_DIR} https://github.com/google/fonts/raw/main/ofl/{path}', shell=True)
SAMPLE_FONTS = [f'{FONT_DIR}/Kalam-Regular.ttf', f'{FONT_DIR}/TiroDevanagariSanskrit-Regular.ttf',
                '/usr/share/fonts/truetype/noto/NotoSerifDevanagari-Regular.ttf']

def degrade(text, font_path, rng):                      # same degradation recipe as Phase 2c
    font = ImageFont.truetype(font_path, 44, layout_engine=ImageFont.Layout.RAQM)
    l, t, r, b = font.getbbox(text); pad = 24
    bg = rng.randint(185, 235)
    paper = np.clip(np.random.default_rng(rng.randint(0, 1 << 30)).normal(bg, 14, (b - t + 2 * pad, r - l + 2 * pad)), 0, 255)
    img = Image.fromarray(paper.astype('uint8'))
    ImageDraw.Draw(img).text((pad - l, pad - t), text, font=font, fill=rng.randint(10, 60))
    img = img.rotate(rng.uniform(-1.5, 1.5), expand=True, fillcolor=bg).filter(ImageFilter.GaussianBlur(rng.uniform(0.6, 1.2)))
    s = rng.uniform(0.45, 0.6); img = img.resize((int(img.width * s), int(img.height * s))).resize(img.size)
    buf = io.BytesIO(); img.save(buf, 'JPEG', quality=rng.randint(30, 60)); return Image.open(buf).convert('L')

SAMPLES = f'{D}/samples'; os.makedirs(SAMPLES, exist_ok=True)
rng = random.Random(2026)
texts = rng.sample([r['clean'] for r in rows_test_rule if 30 <= len(r['clean']) <= 70], 6)   # held-out text, never trained on
samples = []
for k, text in enumerate(texts):
    font = SAMPLE_FONTS[k % len(SAMPLE_FONTS)]
    img = degrade(text, font, rng); img.save(f'{SAMPLES}/sample_{k + 1}.png')
    ocr = norm(pytesseract.image_to_string(img, lang='san', config='--psm 7')).replace('\n', ' ')
    fixed = corr.correct([ocr])[0]
    samples.append({'image': f'samples/sample_{k + 1}.png', 'font': os.path.basename(font), 'clean': text, 'ocr': ocr, 'fixed': fixed,
                    'cer_ocr': jiwer.cer(text, ocr or '-'), 'cer_fixed': jiwer.cer(text, fixed or '-')})
    print(f"[{k + 1}] {os.path.basename(font)}\n  TRUE : {text}\n  OCR  : {ocr}   (CER {samples[-1]['cer_ocr']:.3f})\n  FIXED: {fixed}   (CER {samples[-1]['cer_fixed']:.3f})")
json.dump(samples, open(f'{SAMPLES}/samples.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

[1] Kalam-Regular.ttf
  TRUE : तारप्रेषः पश्चिमदिशि अमेरिका-देशस्य उपनिवेशवादस्य साधनम् अभूत् ।
  OCR  : तारप्रेषः पश्चिमदिशि अमेरिका-दरेशस्त्र उपनितेशताटसप्र साधनम् अभूत् ।   (CER 0.141)
  FIXED: तारप्रेषः पश्चिमदिशि अमेरिका-देशस्य उपनितेशनाटस्य साधनम् अभूत् ।   (CER 0.047)
[2] TiroDevanagariSanskrit-Regular.ttf
  TRUE : २०११ जनगणनानुगुणं मधुरैमण्डलस्य जनसंख्या ३,०४१,०३८
  OCR  : २०१९ जनगणनानुगुणं मधुरैमण्डलस्य जनसंख्या ३,०४९,०३८   (CER 0.040)
  FIXED: २०१९ जनगणनानुगुणं मधुरैमण्डलस्य जनसंख्या ३,०४९,०३८   (CER 0.040)
[3] NotoSerifDevanagari-Regular.ttf
  TRUE : महम्मदीयानां क्रैस्तानां च शैल्याः मिश्रणमपि अत्र दृश्यते ।
  OCR  : महम्मदीयानां क्रैस्तानां च शैल्याः मिश्रणमपि अत्र दृश्यते ।   (CER 0.000)
  FIXED: महम्मदीयानां क्रैस्तानां च शैल्याः मिश्रणमपि अत्र दृश्यते ।   (CER 0.000)
[4] Kalam-Regular.ttf
  TRUE : श्रीसार्वभौमोऽपि महानैयायि केषु परिगण्यते स्म
  OCR  : श्रीसार्वभांमोऽपि महानैमापि केषु परिगण्मते सख   (CER 0.156)
  FIXED: श्रीसार्वभांमोऽपि महानैमापि केषु परिगण्यते स्म   (C

In [ ]:
# 4e. Failure analysis: biggest wins, over-corrections (hallucinations) and lines made worse.
def show(rows, title, n=5):
    print(f'\n=== {title} ({len(rows)}) ===')
    for r in rows[:n]:
        print(f"  TRUE : {r['clean']}\n  OCR  : {r['noisy']}\n  FIXED: {r['pred']}\n")
allr = rows_test_render + rows_test_rule
cer = lambda a, b: jiwer.cer(a, b) if a else float(a != b)
already_ok = [r for r in allr if r['noisy'] == r['clean']]
show([r for r in already_ok if r['pred'] != r['clean']], f'OVER-CORRECTION: input was already correct, model changed it (of {len(already_ok)} correct inputs)')
show([r for r in allr if cer(r['clean'], r['pred']) > cer(r['clean'], r['noisy'])], 'MADE WORSE')
show([r for r in allr if r['noisy'] != r['clean'] and r['pred'] != r['clean']], 'NOT (FULLY) FIXED')
show(sorted([r for r in allr if r['pred'] == r['clean']], key=lambda r: -cer(r['clean'], r['noisy'])), 'BIGGEST FULL FIXES')
# real-word errors from Phase 1 (OCR produced another valid word): compare whole words, not substrings (संयोगः contains योगः)
for w_true, w_ocr in [('रोगः', 'योगः'), ('दोष', 'द्रोण'), ('विहारः', 'निहार')]:
    for i, p in pages.items():
        n_ocr = flat(p['ocr']).split().count(w_ocr)
        if n_ocr:
            print(f"real-word error {w_ocr}->{w_true} on page {i}: {n_ocr} in OCR, {flat(p['fixed']).split().count(w_ocr)} left after correction")

=== OVER-CORRECTION: input was already correct, model changed it (of 388 correct inputs) (43) ===
  TRUE : द्वादशवर्षीयः श्रीज्ञानी केनचित् आत्मघातिरोगेण ग्रस्तः अभवत् ।
  OCR  : द्वादशवर्षीयः श्रीज्ञानी केनचित् आत्मघातिरोगेण ग्रस्तः अभवत् ।
  FIXED: द्वादशवर्षीयः श्रीज्ञानी केनचित् आत्मधातिरोगेण ग्रस्तः अभवत् ।

  TRUE : १.उपायभक्तिः प्रारब्धव्यतिरिक्ताघनाशिनी ।
  OCR  : १.उपायभक्तिः प्रारब्धव्यतिरिक्ताघनाशिनी ।
  FIXED: १.उपायभक्तिः प्रारब्धव्यतिरिक्ताधनाशिनी ।

  TRUE : उद्याने नमदाफा-नदी वहति ।
  OCR  : उद्याने नमदाफा-नदी वहति ।
  FIXED: उद्याने नमदाका-नदी वहति ।

  TRUE : इति निबन्धं विरचय्य पि
  OCR  : इति निबन्धं विरचय्य पि
  FIXED: इति निबन्धं विरचस्य पि

  TRUE : कृतमध्यविलं विलोक्यते धृतगम्भीरखनीखनीलिमा ॥
  OCR  : कृतमध्यविलं विलोक्यते धृतगम्भीरखनीखनीलिमा ॥
  FIXED: कृतमध्यविलं विलोक्यते धृतगम्भीरखनीखनीलिया ॥


=== MADE WORSE (136) ===
  TRUE : काश्मीरदेशे शार्दालप्यामस्यैकमेव पुस्तकमधुनावधि प्राप्तम्।
  OCR  : काश्मीरदेशे शाद्रलिप्यामर्रैकमेव परस्तकमधुनावधि प्राप्तम्।
  FIXE

### Phase 4 summary
| Test set | CER before → after | WER before → after |
|---|---|---|
| **Real page images** (5 pages) | 9.6% → **8.2%** (beam 4: **7.7%**), CIs overlap | 33.9% → 32.2% |
| **Noisy rendered images** (500 lines, real Tesseract) | 8.8% → **7.4%** (Kalam 15.2% → 12.2%) | 32.6% → 27.8% |
| Synthetic rule noise (1,000 lines) | 7.4% → **3.4%** | 36.0% → 18.5% |

- **Synthetic-to-real gap:** −54% CER on our noise model, but only −15–20% on real Tesseract output.
- **Over-correction:** 43 of 388 already-correct lines (11%) were changed, mostly by applying training confusion pairs in reverse (घ→ध, फ→क) to rare words. The real page 2 has कफः → ककः twice.
- **Real-word errors** (योगः, द्रोण, निहार): 0 of 3 fixed; they need domain context.
- **Fixed well:** consonant swaps (−64% synthetic) and conjuncts (−58%). **Barely fixed:** marks (anusvara/visarga, −11%).
- **Where it helps:** only when OCR is poor; on clean fonts (~2% CER) it is neutral to slightly harmful, so gate it by OCR confidence.

Full discussion: `REPORT.md` §6–7.

In [ ]:
# Compact JPEG copies of the sample images (for transfer) + package the report artefacts and download them to this computer.
import shutil, hashlib, io
from google.colab import files
for k in range(1, 7):
    im = Image.open(f'{SAMPLES}/sample_{k}.png').convert('L')
    im = im.resize((640, round(im.height * 640 / im.width)), Image.LANCZOS)
    im.save(f'{SAMPLES}/sample_{k}.jpg', 'JPEG', quality=60, optimize=True)
    print(k, hashlib.md5(open(f'{SAMPLES}/sample_{k}.jpg', 'rb').read()).hexdigest())
shutil.copy(f'{D}/phase4_results.json', SAMPLES)
shutil.make_archive('/content/phase4_artifacts', 'zip', SAMPLES)
files.download('/content/phase4_artifacts.zip')

1 ffdda01fb78f93a0e9d06e40b0dbcfa7
2 61270e1df4834edaa8c9e7deedc5fced
3 493d9b516ccbc1b5080ad3bb620bff50
4 d7fc81e01ff352172b1089f0d8e6e8b4
5 490932c67859dc0c7d3d84e6dda7f633
6 e49450abe749558f81fd2f649d063ab9

In [ ]:
# 4f. Package EVERYTHING for the repo: dataset splits, per-line predictions, full metrics (with 95% CIs), training log.
import os, json, shutil, glob
from google.colab import files
OUT = '/content/repo_export'; shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(f'{OUT}/datasets'); os.makedirs(f'{OUT}/results')
cer1 = lambda a, b: jiwer.cer(a, b) if a else float(a != b)

# 1) dataset splits + render-OCR pairs + Phase 1 OCR output (as generated in Phase 2, from the Drive backup)
for f in ['train.jsonl', 'dev.jsonl', 'test_rule.jsonl', 'test_render.jsonl', 'render_pairs.jsonl', 'phase1.json']:
    shutil.copy(f'{D}/{f}', f'{OUT}/datasets/{f}')

# 2) per-line predictions with flags
def dump_preds(rows, path):
    with open(path, 'w', encoding='utf-8') as fh:
        for r in rows:
            cb, ca = cer1(r['clean'], r['noisy']), cer1(r['clean'], r['pred'])
            fh.write(json.dumps({**r, 'cer_before': round(cb, 4), 'cer_after': round(ca, 4),
                                 'already_correct': r['noisy'] == r['clean'], 'fully_fixed': r['noisy'] != r['clean'] and r['pred'] == r['clean'],
                                 'over_corrected': r['noisy'] == r['clean'] and r['pred'] != r['clean'], 'made_worse': ca > cb},
                                ensure_ascii=False) + '\n')
dump_preds(rows_test_rule, f'{OUT}/results/predictions_test_rule.jsonl')
dump_preds(rows_test_render, f'{OUT}/results/predictions_test_render.jsonl')
json.dump({'pages': pages, 'line_pairs': [{'gt': g, 'ocr': o, 'corrected': c} for g, o, c in line_pairs]},
          open(f'{OUT}/results/predictions_real_pages.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

# 3) metrics with 95% bootstrap CIs (lines resampled) for every test set
def ci(refs, hyps, n=500): return [round(x, 4) for x in bootstrap_cer(refs, hyps, n=n)]
metrics = {'model': MODEL, 'dev_curve': [{'step': h['step'], 'dev_cer': h['eval_cer'], 'dev_wer': h['eval_wer']} for h in state['log_history'] if 'eval_cer' in h]}
for name, rows in (('test_rule', rows_test_rule), ('test_render', rows_test_render)):
    c, n_, p_ = [r['clean'] for r in rows], [r['noisy'] for r in rows], [r['pred'] for r in rows]
    m = dict(results[name]); m.update({'n_lines': len(rows), 'cer_before_ci95': ci(c, n_), 'cer_after_ci95': ci(c, p_),
              'n_already_correct': sum(r['noisy'] == r['clean'] for r in rows),
              'n_over_corrected': sum(r['noisy'] == r['clean'] and r['pred'] != r['clean'] for r in rows)})
    metrics[name] = m
gl, ol, cl = zip(*line_pairs)
metrics['real_pages'] = dict(results['real_pages'], n_line_pairs=len(line_pairs), cer_before_ci95=ci(gl, ol, 1000), cer_after_ci95=ci(gl, cl, 1000),
                             per_page={i: {'cer_before': jiwer.cer(flat(p['gt']), flat(p['ocr'])), 'cer_after': jiwer.cer(flat(p['gt']), flat(p['fixed'])),
                                           'wer_before': jiwer.wer(flat(p['gt']), flat(p['ocr'])), 'wer_after': jiwer.wer(flat(p['gt']), flat(p['fixed']))} for i, p in pages.items()})
metrics['test_render_by_font'] = {f: {'n': v[0], 'cer_before': v[1], 'cer_after': v[2]} for f, v in results['test_render_by_font'].items()}
metrics['beam4'] = results['beam4']
json.dump(metrics, open(f'{OUT}/results/metrics.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
for k in ('test_rule', 'test_render', 'real_pages'):
    m = metrics[k]; print(f"{k:12} CER {m['cer_before']:.3f} {m['cer_before_ci95']} -> {m['cer_after']:.3f} {m['cer_after_ci95']}"
                          + (f"  over-corrected {m['n_over_corrected']}/{m['n_already_correct']}" if 'n_over_corrected' in m else ''))

# 4) training log (learning curve) of the final checkpoint
shutil.copy(f'{D}/byt5/checkpoint-1600/trainer_state.json', f'{OUT}/results/trainer_state_checkpoint-1600.json')
shutil.copy('/content/work/train.log', f'{OUT}/results/train_resumed_from_400.log')

shutil.make_archive('/content/repo_export', 'zip', OUT)
print({f: os.path.getsize(f) // 1024 for f in glob.glob(f'{OUT}/*/*')}, 'KB | zip', os.path.getsize('/content/repo_export.zip') // 1024, 'KB')
files.download('/content/repo_export.zip')

test_rule    CER 0.074 [0.0701, 0.0782] -> 0.034 [0.0316, 0.0374]  over-corrected 32/233
test_render  CER 0.088 [0.0804, 0.0968] -> 0.074 [0.0668, 0.0818]  over-corrected 11/155
real_pages   CER 0.096 [0.0736, 0.1252] -> 0.082 [0.064, 0.1088]
{'/content/repo_export/results/predictions_real_pages.json': 12, '/content/repo_export/results/trainer_state_checkpoint-1600.json': 7, '/content/repo_export/results/predictions_test_rule.jsonl': 547, '/content/repo_export/results/metrics.json': 3, '/content/repo_export/results/train_resumed_from_400.log': 100, '/content/repo_export/results/predictions_test_render.jsonl': 285, '/content/repo_export/datasets/phase1.json': 11, '/content/repo_export/datasets/test_rule.jsonl': 284, '/content/repo_export/datasets/render_pairs.jsonl': 1754, '/content/repo_export/datasets/dev.jsonl': 431, '/content/repo_export/datasets/train.jsonl': 19180, '/content/repo_export/datasets/test_render.jsonl': 156} KB | zip 3583 KB

## Phase 5: Publish the model and reproduce
The final model (`checkpoint-1600`) is published publicly on Hugging Face: **https://huggingface.co/MrunalYewatkar45/byt5-small-sanskrit-post-ocr**

Only what's needed for inference is uploaded (weights, config, tokenizer files, training log, model card, about 1.2 GB). The optimizer and scheduler states are left out.

**Log in safely:** run the login cell and paste a **Write** token from huggingface.co/settings/tokens into the box. The token stays in the Colab session only. It's never written in code and never saved in this notebook.

*To just use the published model, skip to cell 5d, which works in any fresh Colab notebook with no Drive and no login.*

In [ ]:
# 5a. Log in to Hugging Face. Paste a WRITE token into the box below (never put tokens in code: they'd be saved in the notebook).
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 5b. Pre-flight: Drive checkpoint present, Hugging Face login active.
import os
CKPT = '/content/drive/MyDrive/sanskrit_ocr/byt5/checkpoint-1600'
print('drive mounted:', os.path.exists('/content/drive/MyDrive'))
print('checkpoint:', sorted(os.listdir(CKPT)) if os.path.exists(CKPT) else 'NOT FOUND')
try:
    from huggingface_hub import whoami
    print('HF logged in as:', whoami()['name'])
except Exception as e:
    print('HF not logged in:', type(e).__name__)

drive mounted: True
checkpoint: ['added_tokens.json', 'config.json', 'generation_config.json', 'model.safetensors', 'optimizer.pt', 'rng_state.pth', 'scheduler.pt', 'tokenizer_config.json', 'trainer_state.json', 'training_args.bin']
HF logged in as: MrunalYewatkar45

In [ ]:
# 5c. Upload the final model (checkpoint-1600) to a PUBLIC Hugging Face repo, with a model card.
from huggingface_hub import HfApi, whoami
info = whoami()
print('token role:', info.get('auth', {}).get('accessToken', {}).get('role'))
api = HfApi()
REPO_ID = f"{info['name']}/byt5-small-sanskrit-post-ocr"

card = f'''---
language: sa
license: apache-2.0
base_model: google/byt5-small
tags: [ocr, post-ocr-correction, sanskrit, devanagari, byt5, text2text-generation]
pipeline_tag: text2text-generation
---
# ByT5-small: Sanskrit post-OCR correction

Fine-tuned `google/byt5-small` that corrects Tesseract (`san`) output on Sanskrit Devanagari text, one line at a time.
Code, data, evaluation and report: https://github.com/Mrunal31-Stark/OCR_Extraction_indic_text

## Usage
```python
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tok = AutoTokenizer.from_pretrained("{REPO_ID}")
model = AutoModelForSeq2SeqLM.from_pretrained("{REPO_ID}")
x = tok(["आयुः तस्त्र हितं च अदितं च मानम् च"], return_tensors="pt")
print(tok.batch_decode(model.generate(**x, max_length=384), skip_special_tokens=True))
# -> ['आयुः तस्य हितं च अदितं च मानम् च']
```

## Training
- Data: 67,984 (noisy → clean) line pairs from Sanskrit Wikipedia. Noise is rule-based OCR corruption calibrated on real Tesseract errors, plus render-then-OCR pairs (6 fonts).
- Full fine-tune, fp32 on a Colab T4, Adafactor, LR 5e-4, effective batch 32. Checkpoint at step 1,600 of 2,125 (1 epoch), best dev CER 5.2%.

## Results (CER before → after, 95% bootstrap CI)
| Test set | CER |
|---|---|
| 5 real page images (Tesseract output) | 9.6% [7.4–12.5] → 8.2% [6.4–10.9]; beam 4: 7.7% |
| 500 rendered + OCR'd line images | 8.8% [8.0–9.7] → 7.4% [6.7–8.2] |
| 1,000 synthetic-noise lines | 7.4% [7.0–7.8] → 3.4% [3.2–3.7] |

## Limitations
- Gains on real OCR are modest, and the confidence intervals overlap on the small real test set.
- Over-correction: about 11% of already-correct lines get changed, mostly look-alike swaps on rare words.
- Real-word errors (OCR producing another valid word) are not fixed.
- Trained on modern Wikipedia Sanskrit, so it may underperform on classical or domain-specific texts.
- Best used only on low-confidence OCR lines. See the repo's REPORT.md for the full failure analysis.
'''
with open(f'{CKPT}/README.md', 'w', encoding='utf-8') as f:
    f.write(card)

api.create_repo(REPO_ID, repo_type='model', private=False, exist_ok=True)
api.upload_folder(
    folder_path=CKPT, repo_id=REPO_ID, repo_type='model',
    allow_patterns=['model.safetensors', 'config.json', 'generation_config.json', 'tokenizer_config.json',
                    'special_tokens_map.json', 'added_tokens.json', 'trainer_state.json', 'README.md'],
    commit_message='Upload ByT5-small Sanskrit post-OCR corrector (checkpoint-1600)')
print('Uploaded:', f'https://huggingface.co/{REPO_ID}')
print('files:', api.list_repo_files(REPO_ID))

token role: None
Uploaded: https://huggingface.co/MrunalYewatkar45/byt5-small-sanskrit-post-ocr
files: ['.gitattributes', 'README.md', 'added_tokens.json', 'config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'trainer_state.json']

In [ ]:
# Fix model-card tag, then verify: download the model fresh from the Hub and check the usage example's output.
card = open(f'{CKPT}/README.md', encoding='utf-8').read().replace('pipeline_tag: text2text-generation\n', '')
open(f'{CKPT}/README.md', 'w', encoding='utf-8').write(card)
api.upload_file(path_or_fileobj=f'{CKPT}/README.md', path_in_repo='README.md', repo_id=REPO_ID, commit_message='Model card: drop unsupported pipeline tag')

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tok = AutoTokenizer.from_pretrained(REPO_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(REPO_ID)
x = tok(['आयुः तस्त्र हितं च अदितं च मानम् च', 'योगः नाम दोष वेषम्म कारणम्'], return_tensors='pt', padding=True)
print(tok.batch_decode(model.generate(**x, max_length=384), skip_special_tokens=True))

['आयुः तस्य हितं च अदितं च मानम् च', 'योगः नाम दोष वेषम्य कारणम्']

In [ ]:
import requests  # anonymous request (no token) -> proves the repo is public
r = requests.get('https://huggingface.co/api/models/MrunalYewatkar45/byt5-small-sanskrit-post-ocr')
d = r.json(); print(r.status_code, '| private:', d.get('private'), '| files:', [s['rfilename'] for s in d['siblings']])

200 | private: False | files: ['.gitattributes', 'README.md', 'added_tokens.json', 'config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'trainer_state.json']

**5d. Reproduce the results from scratch (what a reviewer would run).** This uses a fresh clone of the GitHub repo and the model from Hugging Face. It needs no Drive, no login and no training. The real-page evaluation reuses the Tesseract output saved in `datasets/phase1.json`, and the result matches Phase 4 exactly (CER 9.65% → 8.23%).

In [ ]:
# Reviewer-style check: fresh clone + HF model, run the documented commands.
!rm -rf /content/fresh && git clone -q https://github.com/Mrunal31-Stark/OCR_Extraction_indic_text.git /content/fresh
!cd /content/fresh && pip -q install -r requirements.txt 2>&1 | tail -1
!cd /content/fresh && python src/infer.py --model MrunalYewatkar45/byt5-small-sanskrit-post-ocr --text "योगः नाम दोष वेषम्म कारणम्" 2>/dev/null
!cd /content/fresh && python src/evaluate.py --model MrunalYewatkar45/byt5-small-sanskrit-post-ocr --real-pages datasets/phase1.json --data datasets/test_render.jsonl --limit 100 --out /content/eval.json 2>/dev/null

योगः नाम दोष वेषम्य कारणम्
datasets/test_render.jsonl: n=100 CER 0.0904 -> 0.0739 | WER 0.3289 -> 0.2639 | fully fixed 10.8% | over-correction 2.9% | made worse 11.0%
  error types {'consonant': 127, 'conjunct': 61, 'mark': 68, 'space/punct': 19} -> {'conjunct': 51, 'mark': 66, 'consonant': 89, 'space/punct': 12}
page 1: CER 0.1212 -> 0.0657
  FIXED: अथ आयुर्वेदः नाम शास्त्रम् आयुर्वेदः नाम आगुः वेटयति इति आयुर्वेद आरूः नाम शरीर इन्दिम सत्त्व आत्म संयोगः तस्य हिताहितं मुखं दृखम् आयुः तस्य हितं च अदितं च मानम् च तच् च यत्र उक्तं तत् आयुर्वेद
page 2: CER 0.0579 -> 0.0744
  FIXED: त्रयो द्रोषाः वातः पित्तं ककः इति एते द्रोणाः शरीरस्य धारणम् कुर्वन्ति तातः गतिः कारणम् पित्तं पाक कारणम् ककः स्थिरत्वं कारणम्
page 3: CER 0.0984 -> 0.1066
  FIXED: आहारः महत्वं आयुः वर्धनम् करोति सम्यक् आहारः शरीरे बलम् वर्धयति असम्यक् आदारः रोगं कारणम् भवति दिनाहार येक्ते आगुः वर्धयति
page 4: CER 0.0970 -> 0.0896
  FIXED: द्वितचर्मा नाम नित्य कर्म अनुष्ठानम् प्रातः उत्थाने शचं दन्त धावने कर्तव्यम् व्यायामः शरी